In [ ]:
import pandas as pd
import re
import os
import os
import re
import torch
from PIL import Image
from transformers import AutoProcessor, CLIPModel
import torch.nn as nn
import os
import re
import torch
import pandas as pd
from PIL import Image
import torch.nn as nn
from transformers import AutoProcessor, CLIPModel, AutoImageProcessor, AutoModel
from google.colab import drive




In [ ]:
# mount drive
drive.mount('/content/drive')

In [ ]:
#strip paths from /content/drive/MyDrive/Thesis/images_generated/bbc_images_0006_139_2.jpg' to ./bbc/images/0006/139.jpg
def strip_path(path):
    filename = path.split('/')[-1]
    parts = filename.replace('.jpg', '').split('_')
    prefix = '_'.join(parts[:-4])
    return f"./{prefix}/images/{parts[-3]}/{parts[-2]}.jpg"

df_generated_images = pd.read_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')
df_generated_images['image_path_stripped'] = df_generated_images['image_path'].apply(strip_path)

grouped_counts = df_generated_images.groupby('image_path_stripped').size().reset_index(name='count')

# get images that have 5 generated images
fullgeneratedimages = grouped_counts[grouped_counts['count'] == 5]
fullgeneratedimages


In [ ]:
def get_images(image_paths):
    if isinstance(image_paths, str):
        image_paths = [image_paths]
    all_images_paths = []

    for image_path in image_paths:
        images_paths = []
        image_path_strip = image_path.strip().lstrip('./')
        # get original path
        original_image_path = f'/content/drive/MyDrive/Thesis/images_dataset/origin/{image_path_strip}'

        # set standard for generated path
        parts = image_path_strip.split('/')
        image_name_path_based = f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}_X.jpg"

        # create base for generated path
        base_generated_path = '/content/drive/MyDrive/Thesis/images_generated/'

        images_paths.append(original_image_path)

        # get all genenerated paths based on standard path
        for i in range(1, 6):
            gen_image_name = image_name_path_based.replace('X', str(i))
            gen_image_path = base_generated_path + gen_image_name
            images_paths.append(gen_image_path)
            print(gen_image_path)

        all_images_paths.extend(images_paths)

    return all_images_paths

# get all paths of generated images
# images_paths = get_images(fullgeneratedimages['image_path_stripped'])
images_paths = get_images('./bbc/images/0004/756.jpg')
images_paths = list(set(images_paths))

In [ ]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

# Load CLIP model
clip_processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# Load DINOv2 model
dino_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino_model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)

cos = nn.CosineSimilarity(dim=0)


In [ ]:
def convert_path(path):
    pattern = r'^/content/drive/MyDrive/Thesis/images_dataset/origin/[^/]+/images/\d{4}/\d+\.jpg$'
    if re.match(pattern, path):
        return path

    filename = os.path.basename(path)  # e.g., bbc_images_0005_442_1.jpg
    match = re.match(r'(\w+)_images_(\d{4})_(\d+)_\d\.jpg$', filename)

    if not match:
        raise ValueError(f"Filename format not recognized: {filename}")

    category, id1, id2 = match.groups()
    new_path = f"/content/drive/MyDrive/Thesis/images_dataset/origin/{category}/images/{id1}/{id2}.jpg"
    return new_path


In [ ]:
def compute_similarities_df(images_paths):
    data = []

    for path in images_paths:
        try:
            real_path = convert_path(path)

            image1 = Image.open(real_path).convert("RGB")
            image2 = Image.open(path).convert("RGB")

            with torch.no_grad():
                # CLIP
                clip_inputs1 = clip_processor(images=image1, return_tensors="pt").to(device)
                clip_inputs2 = clip_processor(images=image2, return_tensors="pt").to(device)

                clip_feat1 = clip_model.get_image_features(**clip_inputs1)
                clip_feat2 = clip_model.get_image_features(**clip_inputs2)

                clip_sim = cos(clip_feat1[0], clip_feat2[0]).item()
                clip_sim = (clip_sim + 1) / 2

                # DINO
                dino_inputs1 = dino_processor(images=image1, return_tensors="pt").to(device)
                dino_inputs2 = dino_processor(images=image2, return_tensors="pt").to(device)

                dino_feat1 = dino_model(**dino_inputs1).last_hidden_state.mean(dim=1)
                dino_feat2 = dino_model(**dino_inputs2).last_hidden_state.mean(dim=1)

                dino_sim = cos(dino_feat1[0], dino_feat2[0]).item()
                dino_sim = (dino_sim + 1) / 2

            data.append({
                'image_path': path,
                'real_image': real_path,
                'clip_similarity': clip_sim,
                'dino_similarity': dino_sim
            })

        except Exception as e:
            print(f"Error processing {path}: {e}")
            data.append({
                'image_path': path,
                'real_image': None,
                'clip_similarity': None,
                'dino_similarity': None
            })

    return pd.DataFrame(data)


## Google Top 5 Benchmark similarity

In [ ]:
def transform_image_path_to_google(image_path):
    match = re.search(r'_([1-5])\.jpg$', image_path)
    if not match:
        return None

    new_path = image_path.replace('/images_generated/', '/images_google/')

    # put _google before the .jpg
    base, ext = os.path.splitext(new_path)
    new_path = base + '_google' + ext

    return new_path


In [ ]:
def compute_similarities_df_google(image_paths):
    data = []

    for path in image_paths:
        try:
            real_path = convert_path(path)
            google_path = transform_image_path_to_google(path)
            image1 = Image.open(real_path).convert("RGB")
            image2 = Image.open(google_path).convert("RGB")

            with torch.no_grad():
                # CLIP
                clip_inputs1 = clip_processor(images=image1, return_tensors="pt").to(device)
                clip_inputs2 = clip_processor(images=image2, return_tensors="pt").to(device)

                clip_feat1 = clip_model.get_image_features(**clip_inputs1)
                clip_feat2 = clip_model.get_image_features(**clip_inputs2)

                clip_sim = cos(clip_feat1[0], clip_feat2[0]).item()
                clip_sim = (clip_sim + 1) / 2

                # DINO
                dino_inputs1 = dino_processor(images=image1, return_tensors="pt").to(device)
                dino_inputs2 = dino_processor(images=image2, return_tensors="pt").to(device)

                dino_feat1 = dino_model(**dino_inputs1).last_hidden_state.mean(dim=1)
                dino_feat2 = dino_model(**dino_inputs2).last_hidden_state.mean(dim=1)

                dino_sim = cos(dino_feat1[0], dino_feat2[0]).item()
                dino_sim = (dino_sim + 1) / 2

            data.append({
                'image_path': path,
                'real_image': real_path,
                'clip_similarity_google': clip_sim,
                'dino_similarity_google': dino_sim
            })

        except Exception as e:
            print(f"Error processing {path}: {e}")
            data.append({
                'image_path': path,
                'real_image': None,
                'clip_similarity_google': None,
                'dino_similarity_google': None
            })

    return pd.DataFrame(data)


In [ ]:
df_simularity = compute_similarities_df(images_paths)

In [ ]:
df_simularity_google = compute_similarities_df_google(images_paths)

# Plots

### Get politicians from generated images

In [ ]:
def generate_links(row):
    path = row['image_path']
    parts = path.strip('./').split('/')
    source = parts[0]
    subfolder = parts[2]
    filename = parts[3]
    name_part = filename.replace('.jpg', '')

    base = f"/content/drive/MyDrive/Thesis/images_generated/{source}_images_{subfolder}_{name_part}"
    links = [f"{base}_{i}.jpg" for i in range(1, 6)]
    return links

df_generated_images = pd.read_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')
df_generated_politicians_images = pd.read_excel('/content/drive/MyDrive/Thesis/generated_politicians.xlsx')
df_generated_politicians_images['generated_image_link'] = df_generated_politicians_images.apply(generate_links, axis=1)
df_generated_images_paths_with_politicians = df_generated_politicians_images.explode('generated_image_link').reset_index(drop=True)

df_generated_images = df_generated_images_paths_with_politicians.rename(columns={
    'generated_image_link': 'image_path',
    'image_path': 'image_path_stripped2'
})

df_generated_images = df_generated_images[['image_path', 'image_path_stripped', 'politician']]

## Plot similarity scores

In [ ]:
df_merged2 = df_simularity.merge(
    df_generated_images,
    how='left',
    on='image_path'
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")
gray_palette = ['#666666', '#bbbbbb']  # Medium and light gray

df_long = df_merged2.melt(
    id_vars='politician',
    value_vars=['clip_similarity', 'dino_similarity'],
    var_name='similarity_type',
    value_name='similarity_score'
)

# create boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_long,
    x='politician',
    y='similarity_score',
    hue='similarity_type',
    palette=gray_palette
)

plt.title('Similarity Scores by Politician', fontsize=14)
plt.xlabel('Politician', fontsize=12)
plt.ylabel('Similarity Score', fontsize=12)
plt.legend(title='Similarity Type')
sns.despine()
plt.tight_layout()
plt.show()


## Plot Google top 5 Similarity scores

In [ ]:
df_merged = df_simularity_google.merge(
    df_generated_images,
    how='left',
    on='image_path'
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
gray_palette = ['#666666', '#bbbbbb']

df_long = df_merged.melt(
    id_vars='politician',
    value_vars=['clip_similarity_google', 'dino_similarity_google'],
    var_name='similarity_type',
    value_name='similarity_score'
)

# Create the box plot
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_long,
    x='politician',
    y='similarity_score',
    hue='similarity_type',
    palette=gray_palette
)

plt.title('Similarity Scores by Politician (Google)', fontsize=14)
plt.xlabel('Politician', fontsize=12)
plt.ylabel('Similarity Score', fontsize=12)
plt.legend(title='Similarity Type')
sns.despine()
plt.tight_layout()
plt.show()
